In [3]:
import torch
from transformers import GPT2LMHeadModel, GPT2Tokenizer, AdamW
from torch.utils.data import DataLoader, Dataset

# 自定义数据集
class TextDataset(Dataset):
    def __init__(self, texts, tokenizer, max_length=128):
        self.tokenizer = tokenizer
        self.data = []
        for text in texts:
            encoding = tokenizer(
                text,
                return_tensors='pt',
                truncation=True,
                max_length=max_length,
                padding='max_length'
            )
            # squeeze去掉batch维度
            self.data.append({
                'input_ids': encoding['input_ids'].squeeze(0),
                'attention_mask': encoding['attention_mask'].squeeze(0)
            })

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        return self.data[idx]

# 初始化模型和分词器
model_name = 'gpt2'
tokenizer = GPT2Tokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token
model = GPT2LMHeadModel.from_pretrained(model_name)
model.train()  # 设置为训练模式

# 准备一些示例文本
texts = [
    "Hello, how are you?",
    "I am a language model.",
    "This is a test sentence.",
    "We are training without using Trainer."
]

# 构造数据集和数据加载器
dataset = TextDataset(texts, tokenizer, max_length=32)
dataloader = DataLoader(dataset, batch_size=2, shuffle=True)

# 设置优化器
optimizer = AdamW(model.parameters(), lr=5e-5)

# 设备设置：如果有GPU则使用GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

# 训练循环
num_epochs = 3
for epoch in range(num_epochs):
    print(f"Epoch {epoch+1}/{num_epochs}")
    for batch in dataloader:
        # 将数据移到设备上
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)

        # GPT2作为自回归语言模型，labels通常与input_ids相同
        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=input_ids
        )
        loss = outputs.loss

        # 清零梯度，反向传播，然后更新参数
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        print(f"Loss: {loss.item():.4f}")


/opt/anaconda3/envs/Huggingface/lib/python3.12/site-packages/transformers/optimization.py:591: FutureWarning: This implementation of AdamW is deprecated and will be removed in a future version. Use the PyTorch implementation torch.optim.AdamW instead, or set `no_deprecation_warning=True` to disable this warning
  warnings.warn(


Epoch 1/3
Loss: 8.2826
Loss: 5.6535
Epoch 2/3
Loss: 3.0425
Loss: 2.2032
Epoch 3/3
Loss: 1.0371
Loss: 1.1141
